# Modelo Pré-treinado — Inferência Directa (sem fine-tuning)
### Testa dois modelos base do TorchXRayVision: DenseNet-121 e ResNet-50

**Propósito:** explorar o que o modelo base já consegue prever antes de qualquer adaptação angolana.  
**Nota:** as doenças detectadas são as do dataset original (CheXpert/NIH) — após fine-tuning com dados angolanos, usar `transfer_learning_multiclasse.ipynb`.

In [ ]:
# !pip install torchxrayvision scikit-image

## Modelo 1 — DenseNet-121 (CheXpert)
Treinado no dataset CheXpert da Stanford. Reconhece ~14 patologias.

In [ ]:
import torchxrayvision as xrv
import skimage.io
import torch
import torchvision

# ── 1. Carregar a imagem ──────────────────────────────────────────────────────
img = skimage.io.imread("00000011_006-acteletasis.png")  # muda para o teu ficheiro

# ── 2. Pré-processamento ──────────────────────────────────────────────────────
img = xrv.datasets.normalize(img, 255)  # converte para [-1024, 1024]

if img.ndim == 3:
    img = img.mean(2)   # RGB → cinza (média simples — válida para raio-X)
img = img[None, ...]    # adiciona dimensão de canal → (1, H, W)

transform = torchvision.transforms.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(224)
])
img = transform(img)
img = torch.from_numpy(img)

# ── 3. Escolha do modelo ──────────────────────────────────────────────────────
# CORRECÇÃO 1 — Escolha documentada do peso e nota sobre sigmoid
# O TorchXRayVision aplica sigmoid INTERNAMENTE no forward() do DenseNet.
# Por isso os outputs já estão em [0, 1] — NÃO aplicar sigmoid manualmente,
# caso contrário aplicas sigmoid duas vezes e os valores ficam errados.
#   sigmoid(sigmoid(x)) ≠ sigmoid(x)  ← erro silencioso, difícil de detectar
#
# Opções de peso disponíveis (escolhe uma):
model = xrv.models.DenseNet(weights="densenet121-res224-chex")        # CheXpert (Stanford)
# model = xrv.models.DenseNet(weights="densenet121-res224-mimic_ch")  # MIMIC-CXR (Boston)
# model = xrv.models.DenseNet(weights="densenet121-res224-all")       # todos os datasets combinados
#
# Para transfer learning angolano, o peso escolhido AQUI deve ser o MESMO
# que está em transfer_learning_multiclasse.ipynb (consistência obrigatória).
model.eval()  # modo inferência — desativa dropout e batchnorm de treino

# ── 4. Inferência ─────────────────────────────────────────────────────────────
with torch.no_grad():
    outputs = model(img[None, ...])  # (1, 1, 224, 224) → batch de 1 imagem

# CORRECÇÃO 1 (continuação) — outputs já são probabilidades [0, 1]
# Sigmoid aplicado internamente pelo modelo. Usar outputs directamente.
# A linha abaixo estava comentada no original porque estava CORRECTA assim:
#   # probs = torch.sigmoid(outputs)  ← NÃO usar — double sigmoid
resultados = dict(zip(model.pathologies, outputs[0].detach().numpy()))

# ── 5. Resultados ─────────────────────────────────────────────────────────────
print("=" * 50)
print("  DENSENET-121 (CheXpert) — Probabilidades")
print("=" * 50)
for doenca, prob in sorted(resultados.items(), key=lambda x: x[1], reverse=True):
    barra = "█" * int(prob * 20)
    print(f"{doenca:<35} {prob:.4f}  {barra}")
print("=" * 50)
print("\nNota: estas patologias são do dataset CheXpert (estrangeiro).")
print("Após fine-tuning com dados angolanos, usar transfer_learning_multiclasse.ipynb")

## Modelo 2 — ResNet-50 (todos os datasets)
Arquitectura diferente (ResNet vs DenseNet). Usa resolução 512×512 em vez de 224×224.  
Útil para comparar com o DenseNet e escolher qual usar como base para o fine-tuning.

In [ ]:
import torchxrayvision as xrv
import skimage.io
import torch
import torchvision

# ── 1. Carregar a imagem ──────────────────────────────────────────────────────
img = skimage.io.imread("00000011_006-acteletasis.png")

# ── 2. Pré-processamento ──────────────────────────────────────────────────────
img = xrv.datasets.normalize(img, 255)

if img.ndim == 3:
    img = img.mean(2)
img = img[None, ...]

# CORRECÇÃO 2 — ResNet usa 512×512, não 224×224
# O notebook original já tinha isto correcto. Mantido e documentado:
# DenseNet-121 → XRayResizer(224)
# ResNet-50    → XRayResizer(512)  ← obrigatório, o modelo foi treinado com esta resolução
# Usar 224 com o ResNet dá resultados errados sem erro visível.
transform = torchvision.transforms.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(512)   # ← 512 obrigatório para ResNet-50
])
img = transform(img)
img = torch.from_numpy(img)

# ── 3. Carregar modelo ────────────────────────────────────────────────────────
# CORRECÇÃO 1 (mesma que DenseNet) — sigmoid já aplicado internamente
# O ResNet do TorchXRayVision também aplica sigmoid no forward().
# Não aplicar sigmoid manualmente.
model = xrv.models.ResNet(weights="resnet50-res512-all")
model.eval()

# ── 4. Inferência ─────────────────────────────────────────────────────────────
with torch.no_grad():
    outputs = model(img[None, ...])

# outputs já são probabilidades [0, 1] — sigmoid interno
resultados = dict(zip(model.pathologies, outputs[0].detach().numpy()))

# ── 5. Resultados ─────────────────────────────────────────────────────────────
print("=" * 50)
print("  RESNET-50 (todos os datasets) — Probabilidades")
print("=" * 50)
for doenca, prob in sorted(resultados.items(), key=lambda x: x[1], reverse=True):
    barra = "█" * int(prob * 20)
    print(f"{doenca:<35} {prob:.4f}  {barra}")
print("=" * 50)

## Comparação e Decisão

| Critério | DenseNet-121 (chex) | ResNet-50 (all) |
|----------|--------------------|-----------------|
| Resolução | 224×224 | 512×512 |
| Dataset de treino | CheXpert (Stanford) | NIH + CheXpert + MIMIC + PadChest |
| Patologias | ~14 | ~18 |
| Memória GPU | Menor | Maior |
| **Recomendado para transfer learning angolano** | **Sim** | Possível, mas mais pesado |

**Usar DenseNet-121 como base para o fine-tuning** — menor, mais rápido, e o notebook `transfer_learning_multiclasse.ipynb` já está configurado para ele.